# Snake-RepairLLaMA — Baseline Evaluation

**Goal:** measure how vanilla `codellama/CodeLlama-7b-Python-hf` (no LoRA, no fine-tuning) performs on the QuixBugs (40 bugs) and BugsInPy (196 bugs) IR4/OR2 evaluation sets — using the **same prompts, sampling protocol, and metrics** that we'll use later for the trained adapter, so results are directly comparable.

**Runtime:** Colab Free T4 (16 GB VRAM). Loads CodeLlama-7B in **8-bit** (~7 GB) — fits comfortably.

**Inference protocol (matches the RepairLLaMA paper):**
- 10 candidate patches per bug
- Sampling: `do_sample=True, temperature=1.0, top_p=0.95`  (NOT beam search)
- `max_new_tokens=256`
- 8-bit quantization

**Cell-by-cell flow:**
1. Setup: clone repo, install deps
2. HF login (paste token or load from Colab secret)
3. Sanity check: load model + generate on 1 bug to verify everything works
4. Run full inference on QuixBugs (~40 bugs × 10 = 400 generations)
5. Run full inference on BugsInPy (~196 bugs × 10 = 1960 generations)
6. Score & print Top-1 / Top-3 / Top-10 exact / AST / compile / buried-fix
7. (Optional) Save results JSONL to Google Drive

## 1. Setup — clone repo & install deps

In [1]:
import os


In [2]:
!pwd

/workspace/snake-repairllama-baseline


In [5]:
!rm -rf /content/snake-repairllama-baseline

In [6]:
import os, sys, subprocess

REPO_URL = "https://github.com/AliSuleman27/snake-repairllama-baseline.git"
REPO_DIR = "/content/snake-repairllama-baseline"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("cwd:", os.getcwd())
print("contents:", os.listdir(REPO_DIR))

cwd: /content/snake-repairllama-baseline
contents: ['scripts', 'notebooks', 'requirements.txt', 'data', 'README.md', '.gitignore', 'src', 'results', '.git']


In [4]:
# Install deps. Colab has torch + transformers preinstalled; we add the rest.
%pip install -q -U "transformers>=4.40" "peft>=0.10" "accelerate>=0.30" "bitsandbytes>=0.43" "datasets>=2.18" tqdm


[notice] A new release of pip is available: 24.2 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. HF login

CodeLlama is gated. Add your HF token as a Colab Secret named `HF_TOKEN`, **or** paste it in the prompt below.

In [6]:
import os

token = None
# try:
#     from google.colab import userdata
#     token = userdata.get("HF_TOKEN")
# except Exception:
#     pass

if not token:
    from getpass import getpass
    token = getpass("Paste your HF token (write or read access): ")

from huggingface_hub import login
login(token=token, add_to_git_credential=False)
print("HF login OK")

Paste your HF token (write or read access):  ········


HF login OK


In [7]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

## 3. Sanity check — load model & test on 1 bug

Run this BEFORE the full eval. Verifies model loads, prompt is well-formed, and a generation looks like a Python snippet (not garbage). ~3 min on T4.

In [8]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [9]:
MODEL_NAME = "codellama/CodeLlama-7b-hf"

In [4]:



tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(load_in_8bit=True, llm_int8_threshold=6.0)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map="auto"
)
model.eval()
print("Model loaded. VRAM:", torch.cuda.memory_allocated() / 1e9, "GB")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded. VRAM: 7.012819968 GB


In [5]:
with open("data/quixbugs_eval.jsonl") as f:
    bug = json.loads(f.readline())

print("=== INPUT (IR4) ===")
print(bug["input"])
print("=== GOLD (OR2) ===")
print(bug["output"])

=== INPUT (IR4) ===
def bitcount(n):
    count = 0
    while n:
# Buggy code:
#         n ^= n - 1
<FILL_ME>
        count += 1
    return count

=== GOLD (OR2) ===
        n &= n - 1



In [6]:
inputs = tokenizer(bug["input"], return_tensors="pt")
print("ids:", inputs.input_ids)
print("max id:", inputs.input_ids.max().item(), "vocab:", model.config.vocab_size)
print("decoded:", tokenizer.decode(inputs.input_ids[0]))

ids: tensor([[    1, 32007,   822,  2586,  2798, 29898, 29876,  1125,    13,  1678,
          2302,   353, 29871, 29900,    13,  1678,  1550,   302, 29901,    13,
         29937, 28209,  1927,   775, 29901,    13, 29937,   308,   302,  6228,
         29922,   302,   448, 29871, 29896,    13, 32008, 29871,    13,  4706,
          2302,  4619, 29871, 29896,    13,  1678,   736,  2302,    13, 32009]])
max id: 32009 vocab: 32016
decoded: <s> <PRE> def bitcount(n):
    count = 0
    while n:
# Buggy code:
#         n ^= n - 1
 <SUF> 
        count += 1
    return count
 <MID>


In [7]:
# Pick the first QuixBugs bug, generate 1 patch, eyeball the output.

inputs = tokenizer(bug["input"], return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(
        **inputs, max_new_tokens=256,
        do_sample=True, temperature=1.0, top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
raw = tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== RAW GENERATION ===")
print(raw)

from src.postprocess import extract_patch
print("=== EXTRACTED (strict) ===")
print(extract_patch(raw, mode="strict"))

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


=== RAW GENERATION ===
# Better code:
        if n & 1:
            count += 1
#        if n % 2 == 1:
#            count += 1
        
        n >>= 1
        if n % 2 == 1:
=== EXTRACTED (strict) ===
# Better code:
        if n & 1:
            count += 1
#        if n % 2 == 1:
#            count += 1
        
        n >>= 1
        if n % 2 == 1:


## 4. Full inference — QuixBugs (40 bugs × 10 samples)

~30 min on T4. Resumes if interrupted (re-run the cell).

In [20]:
# Free the sanity-check model first to make room (we re-load via the helper)
import gc
del model, tokenizer
gc.collect(); torch.cuda.empty_cache()

from src.inference import run_inference

run_inference(
    eval_jsonl="data/quixbugs_eval.jsonl",
    output_jsonl="results/quixbugs_codellama_baseline.jsonl",
    model_name=MODEL_NAME,
    n_samples=10,
    load_in_8bit=True,
)

[inference] Loaded 40 bugs from data/quixbugs_eval.jsonl
[inference] Resuming: 18 bugs already done
[inference] Loading tokenizer: codellama/CodeLlama-7b-hf


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


[inference] Using 8-bit quantization (bitsandbytes)
[inference] Loading model: codellama/CodeLlama-7b-hf


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

bugs:   0%|          | 0/22 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


[inference] Done in 1259.2s (57.2s/bug)
[inference] Wrote results/quixbugs_codellama_baseline.jsonl


In [6]:
import gc
# del model, tokenizer
gc.collect(); torch.cuda.empty_cache()


## 5. Full inference — BugsInPy (196 bugs × 10 samples)

~3-4 hours on T4 free. If your Colab session might disconnect, run with `limit=50` first to get partial numbers, or use the resume-on-rerun behavior.

In [10]:
from src.inference import run_inference

# run_inference(
#     eval_jsonl="data/bugsinpy_eval_verified.jsonl",
#     output_jsonl="results/bugsinpy_codellama_baseline.jsonl",
#     model_name=MODEL_NAME,
#     n_samples=10,
#     load_in_8bit=True,
#     sub_batch_size=2,        # <-- add this
# )

run_inference(
    eval_jsonl="data/bugsinpy_eval_verified.jsonl",
    output_jsonl="results/bugsinpy_codellama_baseline.jsonl",
    model_name=MODEL_NAME,
    n_samples=10,
    load_in_8bit=False,         # <-- FP16, RTX 4090 has the VRAM
    sub_batch_size=10,          # <-- full batch again
    max_new_tokens=128,         # <-- optional, ~2x speedup
)


[inference] Loaded 161 bugs from data/bugsinpy_eval_verified.jsonl
[inference] Resuming: 16 bugs already done
[inference] Loading tokenizer: codellama/CodeLlama-7b-hf


config.json:   0%|          | 0.00/637 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/749 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

bugs:   0%|          | 0/145 [00:00<?, ?it/s]

[inference] Done in 757.8s (5.2s/bug)
[inference] Wrote results/bugsinpy_codellama_baseline.jsonl


## 6. Score & report

Top-1 / Top-3 / Top-10 for **exact-match**, **AST-match**, **compile**, and **buried-fix** (gold appears anywhere in the generation — useful to see if the base model 'knows' the fix even if it can't isolate it cleanly).

In [11]:
from src.metrics import evaluate_file, print_report

qb = evaluate_file(
    inference_jsonl="results/quixbugs_codellama_baseline.jsonl",
    eval_jsonl="data/quixbugs_eval.jsonl",
)
print_report("QuixBugs — vanilla CodeLlama-7B-Python (baseline)", qb)


  QuixBugs — vanilla CodeLlama-7B-Python (baseline)
  Total bugs: 40    Bugs with plausibility data: 0
  Top-1  Exact     :    0 / 40 (  0.0%)
  Top-1  AST       :    0 / 40 (  0.0%)
  Top-1  Compile   :   19 / 40 ( 47.5%)

  Top-3  Exact     :    1 / 40 (  2.5%)
  Top-3  AST       :    0 / 40 (  0.0%)
  Top-3  Compile   :   29 / 40 ( 72.5%)
  Top-3  Buried    :   18 / 40 ( 45.0%)

  Top-10 Exact     :    4 / 40 ( 10.0%)
  Top-10 AST       :    0 / 40 (  0.0%)
  Top-10 Compile   :   36 / 40 ( 90.0%)
  Top-10 Buried    :   28 / 40 ( 70.0%)


In [12]:
from src.metrics import evaluate_file, print_report

bp = evaluate_file(
    inference_jsonl="results/bugsinpy_codellama_baseline.jsonl",
    eval_jsonl="data/bugsinpy_eval.jsonl",
)
print_report("BugsInPy — vanilla CodeLlama-7B-Python (baseline)", bp)


  BugsInPy — vanilla CodeLlama-7B-Python (baseline)
  Total bugs: 161    Bugs with plausibility data: 0
  Top-1  Exact     :    0 / 161 (  0.0%)
  Top-1  AST       :    0 / 161 (  0.0%)
  Top-1  Compile   :   52 / 161 ( 32.3%)

  Top-3  Exact     :    0 / 161 (  0.0%)
  Top-3  AST       :    0 / 161 (  0.0%)
  Top-3  Compile   :   99 / 161 ( 61.5%)
  Top-3  Buried    :    8 / 161 (  5.0%)

  Top-10 Exact     :    0 / 161 (  0.0%)
  Top-10 AST       :    0 / 161 (  0.0%)
  Top-10 Compile   :  128 / 161 ( 79.5%)
  Top-10 Buried    :   15 / 161 (  9.3%)


## 7. (Optional) Save results to Google Drive

Persists `results/*.jsonl` so you can compare against the trained-adapter run later.

In [13]:
from google.colab import drive
import shutil, os

drive.mount("/content/drive")
dest = "/content/drive/MyDrive/snake-repairllama-baseline-results"
os.makedirs(dest, exist_ok=True)
for fn in os.listdir("results"):
    if fn.endswith(".jsonl"):
        shutil.copy(f"results/{fn}", f"{dest}/{fn}")
        print("Saved:", fn)

ModuleNotFoundError: No module named 'google'